In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


BASE_FEATURES = ["P_laser", "v_scan", "d_laser", "t_powder"]
DERIVED_FEATURES = ["linear_energy", "peclet_number", "normalized_enthalpy"]

# v3 physical-state representation:
#   W = melt-pool width [um]
#   R = D/W = melt-pool aspect ratio [-]
#   A = melt-pool cross-sectional area [um^2]
TARGETS = ["melt_width", "aspect_ratio", "melt_area"]
RISK = "balling"


def get_features(include_linear_energy: bool = True) -> list[str]:
    """Return model input features.

    Raw process variables are always retained. Linear energy density P/v is
    an additional derived feature; it does not replace P_laser or v_scan.
    """
    return BASE_FEATURES + (DERIVED_FEATURES if include_linear_energy else [])


FEATURES = get_features(True)


def load_316l_csv(path: str | Path) -> pd.DataFrame:
    """Load the supplied 316L single-track CSV and derive v3 geometry state."""
    path = Path(path)
    df = pd.read_csv(path, header=None, skiprows=1)
    if df.shape[1] != 9:
        raise ValueError(f"Expected 9 columns including ID, got {df.shape[1]}")

    df.columns = [
        "id", "P_laser", "v_scan", "d_laser", "t_powder",
        "melt_width", "melt_depth", "melt_area", "balling",
    ]
    df[df.columns] = df[df.columns].apply(pd.to_numeric, errors="raise")
    df["id"] = df["id"].astype(int)
    df["balling"] = df["balling"].astype(np.float32)

    # J/mm: W / (mm/s) = J/mm.
    df["linear_energy"] = df["P_laser"] / df["v_scan"]

    # Conventions: d_laser is treated as the laser spot DIAMETER; v_scan is
    # converted from mm/s to m/s. These conventions match the normalized
    # enthalpy / Peclet definitions used by the scaling-law literature.
    #
    # Pe = v*d/alpha
    # H* = eta*P / (h_s * sqrt(alpha*d^3*v))
    # where h_s = rho*cp*(Tm-T0) is the sensible enthalpy density needed
    # to heat the material from T0 to the melting temperature.
    rho_316l = 7800.0              # kg/m^3
    cp_316l = 460.0                # J/(kg K)
    alpha_316l = 5.0e-6            # m^2/s, representative 316L value
    Tm_316l = 1678.0               # K
    T0_316l = 293.15               # K
    absorptivity_316l = 0.34
    h_s_vol = rho_316l * cp_316l * (Tm_316l - T0_316l)

    v_mps = np.maximum(df["v_scan"].to_numpy(dtype=float) / 1000.0, 1e-12)
    diameter_m = np.maximum(df["d_laser"].to_numpy(dtype=float) / 1000.0, 1e-12)
    power_w = np.maximum(df["P_laser"].to_numpy(dtype=float), 1e-12)
    df["peclet_number"] = v_mps * diameter_m / alpha_316l
    df["normalized_enthalpy"] = (
        absorptivity_316l * power_w
        / (h_s_vol * np.sqrt(alpha_316l * diameter_m**3 * v_mps))
    )

    # v3 physical representation.
    # Aspect ratio is intentionally computed from the measured geometry rather
    # than predicted independently:
    #     R = D / W
    # It will later be used to reconstruct physical depth as D = W * R.
    width = df["melt_width"].clip(lower=1e-6)
    df["aspect_ratio"] = df["melt_depth"] / width

    if not np.isfinite(df["aspect_ratio"]).all():
        raise ValueError("Non-finite aspect_ratio found in melt-pool data")

    return df


@dataclass
class DataScalers:
    x: StandardScaler
    y: StandardScaler
    features: list[str]
    targets: list[str]



def make_iid_split(
    df: pd.DataFrame,
    seed: int = 42,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Initial IID split. A process-space split can be added as a separate experiment."""
    train, temp = train_test_split(df, test_size=1.0 - train_ratio, random_state=seed)
    relative_val = val_ratio / (1.0 - train_ratio)
    val, test = train_test_split(temp, test_size=1.0 - relative_val, random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)



def fit_scalers(train_df: pd.DataFrame, include_linear_energy: bool = True) -> DataScalers:
    features = get_features(include_linear_energy)
    x = StandardScaler().fit(train_df[features])
    y = StandardScaler().fit(train_df[TARGETS])
    return DataScalers(x=x, y=y, features=features, targets=TARGETS.copy())


class MeltPoolDataset(Dataset):
    def __init__(self, df: pd.DataFrame, scalers: DataScalers):
        self.df = df.reset_index(drop=True)
        self.x = scalers.x.transform(self.df[scalers.features]).astype(np.float32)
        self.y = scalers.y.transform(self.df[scalers.targets]).astype(np.float32)
        self.balling = self.df[RISK].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            "x": torch.from_numpy(self.x[idx]),
            "y": torch.from_numpy(self.y[idx]),
            "balling": torch.tensor(self.balling[idx]),
            "id": int(self.df.iloc[idx]["id"]),
        }


In [ ]:
from __future__ import annotations

import torch
from torch import nn


class DeepSupervisionLoss(nn.Module): # param 가중치 조절
    """
    L = L_data + alpha * L_physics
    """

    def __init__(self, gamma: float = 0.8, state_weights=(1.0, 1.0, 1.0)):
        super().__init__()
        weights = torch.as_tensor(state_weights, dtype=torch.float32)
        if weights.ndim != 1 or weights.numel() != 3:
            raise ValueError("state_weights must contain exactly 3 values: W, R, A")
        if torch.any(weights <= 0):
            raise ValueError("state_weights must be positive")
        self.gamma = gamma
        self.register_buffer("state_weights", weights)

    def forward(self, predictions, target):
        if target.size(-1) != self.state_weights.numel():
            raise ValueError(
                f"Expected target dimension {self.state_weights.numel()}, got {target.size(-1)}"
            )

        total = target.new_tensor(0.0)
        weight_sum = 0.0
        for k, pred in enumerate(predictions):
            loop_weight = self.gamma ** (len(predictions) - 1 - k)
            per_state = (pred - target).pow(2).mean(dim=0)
            state_loss = (per_state * self.state_weights.to(pred)).sum() / self.state_weights.sum()
            total = total + loop_weight * state_loss
            weight_sum += loop_weight
        return total / weight_sum

class PhysicsConsistencyLoss(nn.Module):
    """Mean squared physics residual"""

    def forward(self, feedbacks, confidences=None):
        if not feedbacks:
            return torch.tensor(0.0)

        terms = []
        for i, feedback in enumerate(feedbacks):
            if confidences is not None and confidences[i] is not None:
                mask = confidences[i]
                denom = mask.sum().clamp_min(1.0)
                terms.append((feedback.pow(2) * mask).sum() / denom)
            else:
                terms.append(feedback.pow(2).mean())
        return torch.stack(terms).mean()


In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass

import math
import torch


@dataclass
class PhysicsFeedback:
    residual: torch.Tensor
    confidence: torch.Tensor | None = None


class PhysicsEvaluator(ABC):
    @abstractmethod
    def evaluate(self, process_x: torch.Tensor, predicted_state: torch.Tensor) -> PhysicsFeedback:
        raise NotImplementedError


class ZeroPhysicsEvaluator(PhysicsEvaluator):
    """아키텍처 검증 전용"""

    def evaluate(self, process_x, predicted_state):
        return PhysicsFeedback(residual=torch.zeros_like(predicted_state))


class RosenthalPhysicsEvaluator(PhysicsEvaluator): #Rosenthal의 열역학 모델
    """
    물리 모델:
        W = sqrt( 8/(pi*e) * eta*P / (rho*Cp*v*DeltaT) )
        R = D/W
        A = pi/4 * W * D

    평가:
        L=λW*LW​+λD​*LD​+λA*​LA​
    """

    def __init__(
        self,
        x_mean,
        x_scale,
        y_mean,
        y_scale,
        absorptivity: float = 0.34,
        density: float = 7800.0,
        heat_capacity: float = 460.0,
        thermal_diffusivity: float = 5.0e-6,
        melting_temperature: float = 1678.0,
        preheat_temperature: float = 293.15,
        scaling_k: float = 0.60,
        residual_clip: float = 5.0,
    ):
        self.x_mean = torch.as_tensor(x_mean, dtype=torch.float32)
        self.x_scale = torch.as_tensor(x_scale, dtype=torch.float32)
        self.y_mean = torch.as_tensor(y_mean, dtype=torch.float32)
        self.y_scale = torch.as_tensor(y_scale, dtype=torch.float32)
        self.absorptivity = absorptivity
        self.density = density
        self.heat_capacity = heat_capacity
        self.thermal_diffusivity = thermal_diffusivity
        self.melting_temperature = melting_temperature
        self.preheat_temperature = preheat_temperature
        self.scaling_k = scaling_k
        self.residual_clip = residual_clip

    def _physical_x(self, x: torch.Tensor) -> torch.Tensor:
        mean = self.x_mean.to(device=x.device, dtype=x.dtype)
        scale = self.x_scale.to(device=x.device, dtype=x.dtype)
        return x * scale + mean

    def _physical_y(self, y: torch.Tensor) -> torch.Tensor:
        mean = self.y_mean.to(device=y.device, dtype=y.dtype)
        scale = self.y_scale.to(device=y.device, dtype=y.dtype)
        return y * scale + mean

    def thermal_features(self, process_x: torch.Tensor):
        x = self._physical_x(process_x)
        P = x[:, 0].clamp_min(1e-6)
        v = (x[:, 1] / 1000.0).clamp_min(1e-9)
        diameter = (x[:, 2] / 1000.0).clamp_min(1e-9)

        Pe = v * diameter / self.thermal_diffusivity
        hs_vol = self.density * self.heat_capacity * (
            self.melting_temperature - self.preheat_temperature
        )
        h_star = self.absorptivity * P / (
            hs_vol * torch.sqrt(self.thermal_diffusivity * diameter**3 * v)
        )
        return P, v, diameter, Pe, h_star

    def rosenthal_width(self, process_x: torch.Tensor) -> torch.Tensor:
        """P 및 v 원시 입력을 사용하여 로젠탈 폭을 마이크로미터 단위로 반환합니다."""
        P, v, _, _, _ = self.thermal_features(process_x)
        delta_T = self.melting_temperature - self.preheat_temperature
        width_m = torch.sqrt(
            (8.0 / (math.pi * math.e))
            * self.absorptivity * P
            / (self.density * self.heat_capacity * v * delta_T)
        )
        return width_m * 1e6

    def scaling_depth(self, process_x: torch.Tensor) -> torch.Tensor:
        """마이크로미터 단위의 soft 스케일링 depth 추정치를 반환합니다.

        Using beta = H*/pi and L_th = Pe^(-1/2),
            D / d_laser ~= K * beta * L_th.

        분석용이지 Target은 아님에 주의.
        """
        _, _, diameter, Pe, h_star = self.thermal_features(process_x)
        beta = h_star / math.pi
        normalized_depth = self.scaling_k * beta / torch.sqrt(Pe.clamp_min(1e-12))
        return diameter * normalized_depth * 1e6

    def evaluate(self, process_x: torch.Tensor, predicted_state: torch.Tensor) -> PhysicsFeedback:
        pred = self._physical_y(predicted_state)
        pred_w, pred_r, pred_a = pred.unbind(dim=-1)
        eps = torch.tensor(1e-6, device=pred.device, dtype=pred.dtype)

        ros_w = self.rosenthal_width(process_x)
        phys_d = self.scaling_depth(process_x)
        phys_r = phys_d / pred_w.clamp_min(eps)
        phys_r = phys_r.clamp(min=0.15, max=2.5)
        pred_d = pred_w * pred_r

        r_width = (pred_w - ros_w) / (ros_w.abs() + eps)

        # 로그-비 잔차는 큰 종횡비 값이 피드백을 지배하는 것을 방지하고
        # 기하학적 오차를 대칭적으로 처리합니다.
        raw_r_residual = (
            torch.log(pred_r.clamp_min(1e-4))
            - torch.log(phys_r.clamp_min(1e-4))
        )

        # 분석 법칙이 모델 학습을 압도하지 않도록 기하 신뢰도를 조절합니다.
        R_PHYSICS_CONFIDENCE = 0.25
        r_aspect = R_PHYSICS_CONFIDENCE * raw_r_residual

        # 타원 면적 일관성 식: A ≈ pi/4 * W^2 * R (W * D)
        ellipse_area = math.pi / 4.0 * pred_w.clamp_min(eps) * pred_d.clamp_min(eps)
        r_area = (pred_a - ellipse_area) / (ellipse_area.abs() + eps)

        residual = torch.stack([r_width, r_aspect, r_area], dim=-1)
        residual = torch.clamp(residual, -self.residual_clip, self.residual_clip)

        # Balling 또는 Extreme R 상태(예: R < 0.2 또는 R > 2.0)에서는 단순 타원 가정의 신뢰도를 감쇄합니다.
        regime_penalty = torch.ones_like(pred_r)
        extreme_mask = (pred_r < 0.2) | (pred_r > 2.0)
        regime_penalty[extreme_mask] = 0.1

        # R_PHYSICS_CONFIDENCE 및 면적 일관성 신뢰도에 물리적 영역 패널티를 곱해줍니다.
        confidence_r_tensor = torch.full_like(r_width, R_PHYSICS_CONFIDENCE) * regime_penalty
        confidence_area = torch.ones_like(r_area) * regime_penalty

        confidence = torch.stack(
            [torch.ones_like(r_width), confidence_r_tensor, confidence_area], dim=-1
        )
        return PhysicsFeedback(residual=residual, confidence=confidence)


class DeltaFeedbackEvaluator(PhysicsEvaluator):
    """Debug 전용"""

    def __init__(self, state_dim: int = 3):
        self.state_dim = state_dim
        self.previous = None

    def reset(self):
        self.previous = None

    def evaluate(self, process_x, predicted_state):
        if self.previous is None:
            residual = torch.zeros_like(predicted_state)
        else:
            residual = predicted_state - self.previous
        self.previous = predicted_state.detach()
        return PhysicsFeedback(residual=residual)


In [ ]:
class FluidThermalPhysicsEvaluator(RosenthalPhysicsEvaluator):
    """
    Thermal + Fluid Physics Evaluator.
    Computes Marangoni (Ma), Reynolds (Re), Weber (We), and Capillary (Ca) numbers
    alongside Normalized Enthalpy and Rosenthal Width discrepancies.

    Feedback vector structure to LoopedTransformer:
        [width_discrepancy, aspect_ratio_residual, area_discrepancy, Ma_normalized, Re_normalized, We_normalized, Ca_normalized]
    """
    def __init__(
        self,
        x_mean,
        x_scale,
        y_mean,
        y_scale,
        absorptivity: float = 0.34,
        density: float = 7800.0,              # kg/m^3
        heat_capacity: float = 460.0,          # J/(kg K)
        thermal_diffusivity: float = 5.0e-6,   # m^2/s
        melting_temperature: float = 1678.0,   # K
        preheat_temperature: float = 293.15,   # K
        viscosity: float = 6.0e-3,             # Pa s (liquid metal viscosity)
        surface_tension: float = 1.2,          # N/m
        dsigma_dT: float = -2.6e-4,            # N/(m K) (Marangoni coefficient)
        scaling_k: float = 0.60,
        residual_clip: float = 5.0
    ):
        super().__init__(
            x_mean, x_scale, y_mean, y_scale,
            absorptivity, density, heat_capacity,
            thermal_diffusivity, melting_temperature, preheat_temperature,
            scaling_k, residual_clip
        )
        self.viscosity = viscosity
        self.surface_tension = surface_tension
        self.dsigma_dT = abs(dsigma_dT)

    def evaluate(self, process_x: torch.Tensor, predicted_state: torch.Tensor) -> PhysicsFeedback:
        # 1. Thermal features and physical values reconstruction
        pred = self._physical_y(predicted_state)
        pred_w, pred_r, pred_a = pred.unbind(dim=-1)
        eps = torch.tensor(1e-6, device=pred.device, dtype=pred.dtype)

        P, v, diameter, Pe, h_star = self.thermal_features(process_x)

        # 2. Temperature rise estimation (Delta T)
        # Reference DeltaT = (Tm - T0) scaled with normalized enthalpy to account for power input
        delta_T_ref = self.melting_temperature - self.preheat_temperature
        DeltaT = (delta_T_ref * (1.0 + 0.1 * h_star)).clamp_min(100.0)

        # Melt pool characteristic length (L = melt width in meters)
        L = (pred_w / 1e6).clamp_min(1e-6)

        # 3. Fluid dimensionless numbers calculation
        # Ma = |dsigma_dT| * DeltaT * L / (viscosity * alpha)
        Ma = (self.dsigma_dT * DeltaT * L) / (self.viscosity * self.thermal_diffusivity + eps)

        # Estimated Marangoni velocity U = |dsigma_dT| * DeltaT / viscosity
        U_marangoni = (self.dsigma_dT * DeltaT / (self.viscosity + eps)).clamp_max(10.0) # bound unrealistic speeds

        # Re = rho * U * L / viscosity
        Re = (self.density * U_marangoni * L) / (self.viscosity + eps)

        # We = rho * U^2 * L / surface_tension
        We = (self.density * (U_marangoni ** 2) * L) / (self.surface_tension + eps)

        # Ca = viscosity * U / surface_tension
        Ca = (self.viscosity * U_marangoni) / (self.surface_tension + eps)

        # Rosenthal core evaluations (discrepancy & aspect ratio & area consistency)
        ros_w = self.rosenthal_width(process_x)
        r_width = (pred_w - ros_w) / (ros_w.abs() + eps)

        phys_d = self.scaling_depth(process_x)
        phys_r = (phys_d / pred_w.clamp_min(eps)).clamp(0.15, 2.5)
        raw_r_residual = torch.log(pred_r.clamp_min(1e-4)) - torch.log(phys_r.clamp_min(1e-4))
        r_aspect = 0.25 * raw_r_residual

        pred_d = pred_w * pred_r
        ellipse_area = (math.pi / 4.0) * pred_w.clamp_min(eps) * pred_d.clamp_min(eps)
        r_area = (pred_a - ellipse_area) / (ellipse_area.abs() + eps)

        # 4. Normalize fluid features to avoid high scalar values overpowering gradient update
        Ma_scaled = torch.log10(Ma.clamp_min(1.0)) / 4.0   # target scale: 0 ~ 1 for Ma ~ 10^4
        Re_scaled = torch.log10(Re.clamp_min(0.1)) / 3.0   # target scale
        We_scaled = We.clamp_max(10.0) / 10.0
        Ca_scaled = Ca.clamp_max(0.1) * 10.0

        # 5. Composite Feedback Tensor construction [B, 7]
        residual = torch.stack([
            r_width,
            r_aspect,
            r_area,
            Ma_scaled,
            Re_scaled,
            We_scaled,
            Ca_scaled
        ], dim=-1)
        residual = torch.clamp(residual, -self.residual_clip, self.residual_clip)

        # Geometric and physical regime penalty masking
        regime_penalty = torch.ones_like(pred_r)
        extreme_mask = (pred_r < 0.2) | (pred_r > 2.0)
        regime_penalty[extreme_mask] = 0.1

        # Assign confidence multipliers
        confidence = torch.stack([
            torch.ones_like(r_width),
            torch.full_like(r_width, 0.25) * regime_penalty,
            torch.ones_like(r_area) * regime_penalty,
            torch.full_like(r_width, 0.1),  # fluid numbers are guides
            torch.full_like(r_width, 0.1),
            torch.full_like(r_width, 0.1),
            torch.full_like(r_width, 0.1)
        ], dim=-1)

        return PhysicsFeedback(residual=residual, confidence=confidence)

In [ ]:
class FluidThermalPhysicsEvaluator(RosenthalPhysicsEvaluator):
    """
    Thermal + Fluid Physics Evaluator with explicit progressive loop-wise reasoning:
      - Loop 0: Initial physical-state estimation (No feedback or zero feedback)
      - Loop 1: Thermal consistency refinement (Focus on Rosenthal width discrepancy direction)
      - Loop 2: Fluid/Morphology consistency refinement (Focus on flow regime & aspect ratio discrepancy)
      - Loop 3+: Joint physical consistency refinement (Coupled thermal, fluid, and geometric area consistency)
    """
    def __init__(
        self,
        x_mean,
        x_scale,
        y_mean,
        y_scale,
        absorptivity: float = 0.34,
        density: float = 7800.0,              # kg/m^3
        heat_capacity: float = 460.0,          # J/(kg K)
        thermal_diffusivity: float = 5.0e-6,   # m^2/s
        melting_temperature: float = 1678.0,   # K
        preheat_temperature: float = 293.15,   # K
        viscosity: float = 6.0e-3,             # Pa s
        surface_tension: float = 1.2,          # N/m
        dsigma_dT: float = -2.6e-4,            # N/(m K)
        scaling_k: float = 0.60,
        residual_clip: float = 5.0
    ):
        super().__init__(
            x_mean, x_scale, y_mean, y_scale,
            absorptivity, density, heat_capacity,
            thermal_diffusivity, melting_temperature, preheat_temperature,
            scaling_k, residual_clip
        )
        self.viscosity = viscosity
        self.surface_tension = surface_tension
        self.dsigma_dT = abs(dsigma_dT)

    def evaluate_loop_wise(self, process_x: torch.Tensor, predicted_state: torch.Tensor, loop_idx: int) -> PhysicsFeedback:
        # Unpack predictions in physical scale
        pred = self._physical_y(predicted_state)
        pred_w, pred_r, pred_a = pred.unbind(dim=-1)
        eps = torch.tensor(1e-6, device=pred.device, dtype=pred.dtype)

        # Initialize zero residuals
        r_width = torch.zeros_like(pred_w)
        r_aspect = torch.zeros_like(pred_r)
        r_area = torch.zeros_like(pred_a)
        Ma_scaled = torch.zeros_like(pred_w)
        Re_scaled = torch.zeros_like(pred_w)
        We_scaled = torch.zeros_like(pred_w)
        Ca_scaled = torch.zeros_like(pred_w)

        # Confidence mappings
        conf_width = torch.zeros_like(pred_w)
        conf_aspect = torch.zeros_like(pred_r)
        conf_area = torch.zeros_like(pred_a)
        conf_fluid = torch.zeros_like(pred_w)

        # Loop 0: Initial physical-state estimation (Pure data-driven, zero physics feedback influence)
        if loop_idx == 0:
            residual = torch.stack([r_width, r_aspect, r_area, Ma_scaled, Re_scaled, We_scaled, Ca_scaled], dim=-1)
            confidence = torch.stack([conf_width, conf_aspect, conf_area, conf_fluid, conf_fluid, conf_fluid, conf_fluid], dim=-1)
            return PhysicsFeedback(residual=residual, confidence=confidence)

        # Extract process parameters
        P, v, diameter, Pe, h_star = self.thermal_features(process_x)

        # Loop 1: Thermal consistency refinement
        if loop_idx == 1:
            ros_w = self.rosenthal_width(process_x)
            # Directional correction: positive if pred_w is smaller than thermal scale, negative if larger
            r_width = (ros_w - pred_w) / (ros_w.abs() + eps)
            conf_width = torch.ones_like(r_width)

        # Loop 2: Fluid/Morphology consistency refinement
        elif loop_idx == 2:
            # Calculate fluid dimensionless numbers for guidance
            delta_T_ref = self.melting_temperature - self.preheat_temperature
            DeltaT = (delta_T_ref * (1.0 + 0.1 * h_star)).clamp_min(100.0)
            L = (pred_w / 1e6).clamp_min(1e-6)

            Ma = (self.dsigma_dT * DeltaT * L) / (self.viscosity * self.thermal_diffusivity + eps)
            U_marangoni = (self.dsigma_dT * DeltaT / (self.viscosity + eps)).clamp_max(10.0)
            Re = (self.density * U_marangoni * L) / (self.viscosity + eps)
            We = (self.density * (U_marangoni ** 2) * L) / (self.surface_tension + eps)
            Ca = (self.viscosity * U_marangoni) / (self.surface_tension + eps)

            Ma_scaled = torch.log10(Ma.clamp_min(1.0)) / 4.0
            Re_scaled = torch.log10(Re.clamp_min(0.1)) / 3.0
            We_scaled = We.clamp_max(10.0) / 10.0
            Ca_scaled = Ca.clamp_max(0.1) * 10.0

            # Directional correction for aspect ratio based on fluid dynamics scaling
            phys_d = self.scaling_depth(process_x)
            phys_r = (phys_d / pred_w.clamp_min(eps)).clamp(0.15, 2.5)
            # Correction signal: positive if pred_r is smaller than physical aspect ratio, negative if larger
            r_aspect = torch.log(phys_r.clamp_min(1e-4)) - torch.log(pred_r.clamp_min(1e-4))

            # Apply regime penalty to confidence
            regime_penalty = torch.ones_like(pred_r)
            extreme_mask = (pred_r < 0.2) | (pred_r > 2.0)
            regime_penalty[extreme_mask] = 0.1

            conf_aspect = torch.full_like(r_aspect, 0.25) * regime_penalty
            conf_fluid = torch.full_like(r_width, 0.1)

        # Loop 3+: Joint physical consistency refinement (Coupled loop)
        else:
            # 1. Thermal width discrepancy
            ros_w = self.rosenthal_width(process_x)
            r_width = (ros_w - pred_w) / (ros_w.abs() + eps)

            # 2. Fluid numbers & aspect ratio discrepancy
            delta_T_ref = self.melting_temperature - self.preheat_temperature
            DeltaT = (delta_T_ref * (1.0 + 0.1 * h_star)).clamp_min(100.0)
            L = (pred_w / 1e6).clamp_min(1e-6)
            Ma = (self.dsigma_dT * DeltaT * L) / (self.viscosity * self.thermal_diffusivity + eps)
            U_marangoni = (self.dsigma_dT * DeltaT / (self.viscosity + eps)).clamp_max(10.0)
            Re = (self.density * U_marangoni * L) / (self.viscosity + eps)
            We = (self.density * (U_marangoni ** 2) * L) / (self.surface_tension + eps)
            Ca = (self.viscosity * U_marangoni) / (self.surface_tension + eps)

            Ma_scaled = torch.log10(Ma.clamp_min(1.0)) / 4.0
            Re_scaled = torch.log10(Re.clamp_min(0.1)) / 3.0
            We_scaled = We.clamp_max(10.0) / 10.0
            Ca_scaled = Ca.clamp_max(0.1) * 10.0

            phys_d = self.scaling_depth(process_x)
            phys_r = (phys_d / pred_w.clamp_min(eps)).clamp(0.15, 2.5)
            r_aspect = torch.log(phys_r.clamp_min(1e-4)) - torch.log(pred_r.clamp_min(1e-4))

            # 3. Geometric area consistency
            pred_d = pred_w * pred_r
            ellipse_area = (math.pi / 4.0) * pred_w.clamp_min(eps) * pred_d.clamp_min(eps)
            # Correction signal: positive if pred_a is smaller than geometric ellipse area, negative if larger
            r_area = (ellipse_area - pred_a) / (ellipse_area.abs() + eps)

            regime_penalty = torch.ones_like(pred_r)
            extreme_mask = (pred_r < 0.2) | (pred_r > 2.0)
            regime_penalty[extreme_mask] = 0.1

            conf_width = torch.ones_like(r_width)
            conf_aspect = torch.full_like(r_aspect, 0.25) * regime_penalty
            conf_area = torch.ones_like(r_area) * regime_penalty
            conf_fluid = torch.full_like(r_width, 0.1)

        # Stack and construct feedback
        residual = torch.stack([r_width, r_aspect, r_area, Ma_scaled, Re_scaled, We_scaled, Ca_scaled], dim=-1)
        residual = torch.clamp(residual, -self.residual_clip, self.residual_clip)

        confidence = torch.stack([conf_width, conf_aspect, conf_area, conf_fluid, conf_fluid, conf_fluid, conf_fluid], dim=-1)

        return PhysicsFeedback(residual=residual, confidence=confidence)

    def evaluate(self, process_x: torch.Tensor, predicted_state: torch.Tensor) -> PhysicsFeedback:
        # Default fallback to coupled joint evaluation
        return self.evaluate_loop_wise(process_x, predicted_state, loop_idx=3)

In [ ]:
# 2026. 601Solutions
# 유의해야할 점: 기존 Transformer 모델과 달리 이 Proj의 Transformer의 sequence는 데이터의 시간축이 아니라 추론 iteration이다.

from __future__ import annotations

from dataclasses import dataclass
import torch
from torch import nn

@dataclass
class LoopOutput:
    predictions: list[torch.Tensor]
    feedbacks: list[torch.Tensor]
    confidences: list[torch.Tensor | None] | None = None

class FeatureTokenizer(nn.Module):
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.scalar_projection = nn.Linear(1, d_model)
        self.feature_embedding = nn.Parameter(torch.randn(n_features, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.scalar_projection(x.unsqueeze(-1))
        return tokens + self.feature_embedding.unsqueeze(0)

def make_encoder(d_model: int, n_heads: int, n_layers: int, dropout: float):
    layer = nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=n_heads,
        dim_feedforward=4 * d_model,
        dropout=dropout,
        activation="gelu",
        batch_first=True,
        norm_first=True,
    )
    return nn.TransformerEncoder(layer, num_layers=n_layers)

class LoopedTransformer(nn.Module):
    def __init__(
        self,
        n_features: int = 7,
        state_dim: int = 3,
        feedback_dim: int = 7,  # Default dimension is now 7 to support FluidThermal feedback
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.1,
        loop_steps: int = 3,
    ):
        super().__init__()
        self.state_dim = state_dim
        self.feedback_dim = feedback_dim
        self.d_model = d_model
        self.loop_steps = loop_steps

        self.tokenizer = FeatureTokenizer(n_features, d_model)
        self.state_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        self.transformer = make_encoder(d_model, n_heads, n_layers, dropout)

        self.state_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, state_dim),
        )

        self.feedback_encoder = nn.Sequential(
            nn.Linear(feedback_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.state_update = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(
        self,
        x: torch.Tensor,
        physics_evaluator=None,
        loop_steps: int | None = None,
    ) -> LoopOutput:
        bsz = x.size(0)
        steps = loop_steps or self.loop_steps

        process_tokens = self.tokenizer(x)
        state = self.state_token.expand(bsz, -1, -1)

        predictions = []
        feedbacks = []
        confidences = []

        for _ in range(steps):
            tokens = torch.cat([process_tokens, state], dim=1)
            encoded = self.transformer(tokens)

            state_hidden = encoded[:, -1, :]
            prediction = self.state_head(state_hidden)
            predictions.append(prediction)

            if physics_evaluator is None:
                # default matching shape
                feedback = torch.zeros(bsz, self.feedback_dim, device=x.device, dtype=x.dtype)
                confidence = None
            else:
                result = physics_evaluator.evaluate(x, prediction)
                feedback = result.residual
                confidence = result.confidence

            feedbacks.append(feedback)
            confidences.append(confidence)

            feedback_hidden = self.feedback_encoder(feedback)
            PHYSICS_FEEDBACK_SCALE = 0.25
            state = state + PHYSICS_FEEDBACK_SCALE * (self.state_update(
                torch.cat([state_hidden, feedback_hidden], dim=-1)
            )).unsqueeze(1)

        return LoopOutput(predictions=predictions, feedbacks=feedbacks, confidences=confidences)

In [52]:
class LoopedTransformer(nn.Module):
    def __init__(
        self,
        n_features: int = 7,
        state_dim: int = 3,
        feedback_dim: int = 7,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.1,
        loop_steps: int = 4,
    ):
        super().__init__()
        self.state_dim = state_dim
        self.feedback_dim = feedback_dim
        self.d_model = d_model
        self.loop_steps = loop_steps

        self.tokenizer = FeatureTokenizer(n_features, d_model)
        self.state_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        self.transformer = make_encoder(d_model, n_heads, n_layers, dropout)

        self.state_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, state_dim),
        )

        self.feedback_encoder = nn.Sequential(
            nn.Linear(feedback_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.state_update = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(
        self,
        x: torch.Tensor,
        physics_evaluator=None,
        loop_steps: int | None = None,
    ) -> LoopOutput:
        bsz = x.size(0)
        steps = loop_steps or self.loop_steps

        process_tokens = self.tokenizer(x)
        state = self.state_token.expand(bsz, -1, -1)

        predictions = []
        feedbacks = []
        confidences = []

        for step_idx in range(steps):
            tokens = torch.cat([process_tokens, state], dim=1)
            encoded = self.transformer(tokens)

            state_hidden = encoded[:, -1, :]
            prediction = self.state_head(state_hidden)
            predictions.append(prediction)

            if physics_evaluator is None:
                feedback = torch.zeros(bsz, self.feedback_dim, device=x.device, dtype=x.dtype)
                confidence = None
            else:
                # Explicitly pass loop_idx to implement progressive, sequential physical reasoning
                if hasattr(physics_evaluator, 'evaluate_loop_wise'):
                    result = physics_evaluator.evaluate_loop_wise(x, prediction, loop_idx=step_idx)
                else:
                    result = physics_evaluator.evaluate(x, prediction)
                feedback = result.residual
                confidence = result.confidence

            feedbacks.append(feedback)
            confidences.append(confidence)

            feedback_hidden = self.feedback_encoder(feedback)
            PHYSICS_FEEDBACK_SCALE = 0.25
            state = state + PHYSICS_FEEDBACK_SCALE * (self.state_update(
                torch.cat([state_hidden, feedback_hidden], dim=-1)
            )).unsqueeze(1)

        return LoopOutput(predictions=predictions, feedbacks=feedbacks, confidences=confidences)

In [ ]:
from __future__ import annotations

import argparse
from pathlib import Path
import torch
from torch.utils.data import DataLoader

def build_model(args, n_features: int):
    common = dict(
        n_features=n_features,
        d_model=args.d_model,
        n_heads=args.n_heads,
        n_layers=args.n_layers,
        dropout=args.dropout,
    )
    # Configure feedback dimension depending on whether fluid physics is selected
    f_dim = 7 if args.physics == "fluid" else 3
    return LoopedTransformer(loop_steps=args.loop_steps, feedback_dim=f_dim, **common)

def parse_state_weights(values):
    if len(values) != 3:
        raise ValueError("--state-weights requires exactly 3 values: W R A")
    weights = tuple(float(v) for v in values)
    if any(v <= 0 for v in weights):
        raise ValueError("All state weights must be positive")
    return weights

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="/content/316L_meltpool.csv")
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--loop-steps", type=int, default=10)
    parser.add_argument("--model", choices=["single", "looped"], default="looped")
    parser.add_argument("--physics", choices=["none", "rosenthal", "fluid"], default="fluid")
    parser.add_argument("--physics-alpha", type=float, default=0.1) # set to 0.1 to let physics guide the network representation
    parser.add_argument("--linear-energy", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--state-weights", nargs=3, default=("1.0", "1.0", "1.0"), metavar=("W", "R", "A"))
    parser.add_argument("--gamma", type=float, default=0.8)
    parser.add_argument("--patience", type=int, default=20)
    parser.add_argument("--selection", choices=["data", "total"], default="data")
    parser.add_argument("--physics-feedback-scale",type=float,default=0.25)
    parser.add_argument("--physics-r-confidence", type=float, default=0.25)
    parser.add_argument("--d-model", type=int, default=64)
    parser.add_argument("--n-heads", type=int, default=4)
    parser.add_argument("--n-layers", type=int, default=2)
    parser.add_argument("--dropout", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out", default="looped_transformer_best.pt")
    args, unknown = parser.parse_known_args()
    args.state_weights = parse_state_weights(args.state_weights)

    torch.manual_seed(args.seed)
    df = load_316l_csv(args.data)
    train_df, val_df, test_df = make_iid_split(df, seed=args.seed)
    scalers = fit_scalers(train_df, include_linear_energy=args.linear_energy)

    train_loader = DataLoader(
        MeltPoolDataset(train_df, scalers), batch_size=args.batch_size, shuffle=True
    )
    val_loader = DataLoader(MeltPoolDataset(val_df, scalers), batch_size=args.batch_size)

    model = build_model(args, n_features=len(scalers.features))
    if args.physics == "fluid" and args.model == "looped":
        physics = FluidThermalPhysicsEvaluator(
            x_mean=scalers.x.mean_, x_scale=scalers.x.scale_,
            y_mean=scalers.y.mean_, y_scale=scalers.y.scale_,
        )
    elif args.physics == "rosenthal" and args.model == "looped":
        physics = RosenthalPhysicsEvaluator(
            x_mean=scalers.x.mean_, x_scale=scalers.x.scale_,
            y_mean=scalers.y.mean_, y_scale=scalers.y.scale_,
        )
    else:
        physics = ZeroPhysicsEvaluator()

    data_loss_fn = DeepSupervisionLoss(gamma=args.gamma, state_weights=args.state_weights)
    physics_loss_fn = PhysicsConsistencyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    best_metric = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, args.epochs + 1):
        model.train()
        train_total = train_data = train_phys = 0.0

        for batch in train_loader:
            optimizer.zero_grad()
            out = model(batch["x"], physics_evaluator=physics)
            data_loss = data_loss_fn(out.predictions, batch["y"])
            phys_loss = physics_loss_fn(out.feedbacks, out.confidences)
            loss = data_loss + args.physics_alpha * phys_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            n = batch["x"].size(0)
            train_total += loss.item() * n
            train_data += data_loss.item() * n
            train_phys += phys_loss.item() * n

        train_total /= len(train_loader.dataset)
        train_data /= len(train_loader.dataset)
        train_phys /= len(train_loader.dataset)

        model.eval()
        val_total = val_data = val_phys = 0.0
        with torch.no_grad():
            for batch in val_loader:
                out = model(batch["x"], physics_evaluator=physics)
                data_loss = data_loss_fn(out.predictions, batch["y"])
                phys_loss = physics_loss_fn(out.feedbacks, out.confidences)
                loss = data_loss + args.physics_alpha * phys_loss
                n = batch["x"].size(0)
                val_total += loss.item() * n
                val_data += data_loss.item() * n
                val_phys += phys_loss.item() * n

        val_total /= len(val_loader.dataset)
        val_data /= len(val_loader.dataset)
        val_phys /= len(val_loader.dataset)

        selection_metric = val_data if args.selection == "data" else val_total
        if selection_metric < best_metric:
            best_metric = selection_metric
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "x_scaler": scalers.x,
                    "y_scaler": scalers.y,
                    "features": scalers.features,
                    "targets": scalers.targets,
                    "config": vars(args),
                    "state_representation": "[melt_width, aspect_ratio=D/W, melt_area]",
                },
                out_path,
            )
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:03d} train={train_total:.5f} "
                f"val={val_total:.5f} data_val={val_data:.5f} physics_val={val_phys:.5f}"
            )

        if args.patience > 0 and epochs_without_improvement >= args.patience:
            print(f"Early stopping at epoch {epoch} (patience={args.patience}).")
            break

    print("Training complete.")
    print(f"Best validation {args.selection} objective: {best_metric:.5f} at epoch {best_epoch}")
    print(f"model={args.model}, physics={args.physics}, physics_alpha={args.physics_alpha}")
    print(f"state_representation=[W, R=D/W, A], state_weights={args.state_weights}")
    print(f"features={scalers.features}")
    print("NOTE: FluidThermalPhysicsEvaluator includes Marangoni (Ma), Reynolds (Re), Weber (We), and Capillary (Ca) numbers.")

if __name__ == "__main__":
    main()

/tmp/ipykernel_1360/3023299187.py:36: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=n_layers)


epoch=001 train=1.16559 val=0.83032 data_val=0.80824 physics_val=0.22079
epoch=010 train=0.29056 val=0.26491 data_val=0.24584 physics_val=0.19074
epoch=020 train=0.20824 val=0.25217 data_val=0.23653 physics_val=0.15639
epoch=030 train=0.19584 val=0.20807 data_val=0.19256 physics_val=0.15505
epoch=040 train=0.17089 val=0.19347 data_val=0.17844 physics_val=0.15026
epoch=050 train=0.17655 val=0.18754 data_val=0.17350 physics_val=0.14040
epoch=060 train=0.14015 val=0.15327 data_val=0.13814 physics_val=0.15138
epoch=070 train=0.13997 val=0.18429 data_val=0.17007 physics_val=0.14224
epoch=080 train=0.13809 val=0.13753 data_val=0.12273 physics_val=0.14803
Early stopping at epoch 84 (patience=20).
Training complete.
Best validation data objective: 0.11830 at epoch 64
model=looped, physics=fluid, physics_alpha=0.1
state_representation=[W, R=D/W, A], state_weights=(1.0, 1.0, 1.0)
features=['P_laser', 'v_scan', 'd_laser', 't_powder', 'linear_energy', 'peclet_number', 'normalized_enthalpy']
NOTE: 

In [51]:
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. 저장된 체크포인트 로드
checkpoint_path = "looped_transformer_best.pt"
checkpoint = torch.load(checkpoint_path, weights_only=False)

# 2. 데이터셋 로드 및 분할
df = load_316l_csv("/content/316L_meltpool.csv")
train_df, val_df, test_df = make_iid_split(df, seed=42)

scaler_x = checkpoint["x_scaler"]
scaler_y = checkpoint["y_scaler"]

# 3. 모델 초기화 및 가중치 적용
model = LoopedTransformer(n_features=len(FEATURES), feedback_dim=7, loop_steps=10)
model.load_state_dict(checkpoint["model"])
model.eval()

# 4. 검증 데이터 준비
x_scaled = scaler_x.transform(val_df[FEATURES]).astype(np.float32)
x_tensor = torch.from_numpy(x_scaled)

# 5. 예측 수행
with torch.no_grad():
    output = model(x_tensor)
    final_predictions_scaled = output.predictions[-1].numpy()

# 6. 역변환 (Inverse Transform)을 통한 원래 스케일 복원
final_predictions = scaler_y.inverse_transform(final_predictions_scaled)

# 7. 실제값과 예측값 비교용 DataFrame 생성
comparison_df = pd.DataFrame()
for i, target_col in enumerate(TARGETS):
    comparison_df[f"Actual_{target_col}"] = val_df[target_col]
    comparison_df[f"Pred_{target_col}"] = final_predictions[:, i]
    comparison_df[f"Error_{target_col}"] = comparison_df[f"Actual_{target_col}"] - comparison_df[f"Pred_{target_col}"]

# 기하적 일관성 지표 계산 (A_pred vs pi/4 * W_pred^2 * R_pred)
pred_w = comparison_df["Pred_melt_width"]
pred_r = comparison_df["Pred_aspect_ratio"]
pred_a = comparison_df["Pred_melt_area"]

ellipse_area_pred = (np.pi / 4.0) * pred_w * (pred_w * pred_r)
geometric_residual = np.abs(pred_a - ellipse_area_pred)
relative_geometric_residual = geometric_residual / pred_a.clip(lower=1e-6)

print("=== Validation Set Performance Metrics ===")
for target_col in TARGETS:
    act = comparison_df[f"Actual_{target_col}"]
    prd = comparison_df[f"Pred_{target_col}"]
    mae = mean_absolute_error(act, prd)
    rmse = np.sqrt(mean_squared_error(act, prd))
    print(f"{target_col:<15} | MAE: {mae:8.4f} | RMSE: {rmse:8.4f}")

print("\n=== Geometric Consistency Indicators ===")
print(f"Mean Absolute Area Residual (|A_pred - pi/4 * W^2 * R|): {geometric_residual.mean():.4f} um^2")
print(f"Mean Relative Geometric Error: {relative_geometric_residual.mean() * 100:.4f}%")

print("\n--- First 5 Prediction Samples ---")
display(comparison_df[[f"Actual_{col}" for col in TARGETS] + [f"Pred_{col}" for col in TARGETS]].head(5))

/tmp/ipykernel_1360/3023299187.py:36: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=n_layers)


=== Validation Set Performance Metrics ===
melt_width      | MAE:  43.5364 | RMSE:  55.2337
aspect_ratio    | MAE:   0.4008 | RMSE:   0.5556
melt_area       | MAE: 10942.6388 | RMSE: 16017.7576

=== Geometric Consistency Indicators ===
Mean Absolute Area Residual (|A_pred - pi/4 * W^2 * R|): 3974.3667 um^2
Mean Relative Geometric Error: 22.6459%

--- First 5 Prediction Samples ---


,Actual_melt_width,Actual_aspect_ratio,Actual_melt_area,Pred_melt_width,Pred_aspect_ratio,Pred_melt_area
0,117.300,0.824382,9203.400,166.815384,0.476575,14207.372070
1,232.600,1.136285,56199.400,226.426987,1.547997,56144.339844
2,85.900,0.665891,4268.700,157.148300,0.080708,3197.573242
3,111.900,2.023235,20580.300,104.288963,2.227400,26946.468750
4,109.651,0.986557,8134.272,168.977570,0.230527,6543.374023
